# 🛡️ FATI — Entraînement du Modèle NLP Anti-Phishing & Fake Content
## iSAFE Hackathon SMSI 2026 — Mouhammadou DIA

---

### 📌 Ce que fait ce notebook :
1. **Télécharge** les datasets FakeNewsNet (CSV) + UCI Phishing
2. **Prépare** et fusionne les données (features textuelles + structurelles)
3. **Entraîne** deux modèles :
   - `Random Forest` sur features structurelles (rapide, interprétable)
   - `DistilBERT fine-tuné` sur le texte (NLP profond)
4. **Évalue** les modèles (F1, AUC-ROC, matrice de confusion)
5. **Exporte** les deux modèles au format **ONNX** pour l'extension Chrome

### ⚙️ Environnement recommandé :
- Google Colab avec **GPU T4** (Runtime → Modifier le type d'exécution → GPU)
- RAM : 12 GB minimum

---

## 📦 ÉTAPE 0 — Installation des dépendances

In [ ]:
%%capture
!pip install transformers==4.40.0
!pip install datasets==2.19.0
!pip install onnx==1.16.0
!pip install onnxruntime==1.18.0
!pip install optimum[onnxruntime]==1.19.0
!pip install scikit-learn==1.4.2
!pip install skl2onnx==0.5.0
!pip install imbalanced-learn==0.12.2
!pip install nltk==3.8.1
!pip install tldextract==5.1.2
!pip install seaborn matplotlib
print('✅ Toutes les dépendances sont installées.')

## 📥 ÉTAPE 1 — Téléchargement des Datasets

In [ ]:
import pandas as pd
import numpy as np
import requests
import os
import zipfile
import io

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('models/onnx', exist_ok=True)

print('📁 Dossiers créés.')

In [ ]:
# ─── 1.1  FakeNewsNet (CSV minimal depuis GitHub) ────────────────────────────
BASE = 'https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset/'

files = {
    'politifact_fake.csv' : BASE + 'politifact_fake.csv',
    'politifact_real.csv' : BASE + 'politifact_real.csv',
    'gossipcop_fake.csv'  : BASE + 'gossipcop_fake.csv',
    'gossipcop_real.csv'  : BASE + 'gossipcop_real.csv',
}

fnn_dfs = []
for fname, url in files.items():
    df = pd.read_csv(url)
    df['label_fnn'] = 1 if 'fake' in fname else 0   # 1 = fake/phishing
    df['source']    = fname.split('_')[0]             # politifact | gossipcop
    fnn_dfs.append(df)
    print(f'  ✅ {fname} — {len(df)} lignes')

fnn = pd.concat(fnn_dfs, ignore_index=True)
print(f'\n📊 FakeNewsNet total : {len(fnn)} entrées')
print(fnn['label_fnn'].value_counts().rename({0:'real', 1:'fake'}))

In [ ]:
# ─── 1.2  UCI Phishing Websites Dataset ──────────────────────────────────────
# Source officielle : UCI ML Repository
uci_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00327/Training%20Dataset.arff'

# Téléchargement et parsing ARFF manuel (léger, sans dépendance scipy)
resp = requests.get(uci_url, timeout=30)
lines = resp.text.splitlines()

# Extraction des noms de colonnes
attrs = [l.split()[1] for l in lines if l.upper().startswith('@ATTRIBUTE')]

# Extraction des données
data_start = next(i for i, l in enumerate(lines) if l.upper().startswith('@DATA')) + 1
data_lines = [l for l in lines[data_start:] if l.strip() and not l.startswith('%')]

uci = pd.DataFrame(
    [row.split(',') for row in data_lines],
    columns=attrs
).apply(pd.to_numeric, errors='coerce')

# La colonne cible s'appelle 'Result' : -1 = phishing, 1 = légitime
uci['label_phishing'] = uci['Result'].map({-1: 1, 1: 0})  # 1 = phishing
uci.drop(columns=['Result'], inplace=True)

print(f'📊 UCI Phishing Dataset : {len(uci)} entrées, {uci.shape[1]} features')
print(uci['label_phishing'].value_counts().rename({0:'légitime', 1:'phishing'}))

## 🔧 ÉTAPE 2 — Préparation des Features

In [ ]:
import re
import tldextract
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

STOP_EN = set(stopwords.words('english'))
STOP_FR = set(stopwords.words('french'))
STOP = STOP_EN | STOP_FR

# ── Lexique de phishing / contenu trompeur ───────────────────────────────────
PHISHING_KEYWORDS = [
    # Urgence
    'urgent', 'immediately', 'account suspended', 'verify now', 'expires',
    'limited time', 'act now', 'warning', 'alert', 'suspended',
    # Demandes sensibles
    'password', 'credit card', 'social security', 'bank account', 'otp',
    'confirm your', 'update your', 'login', 'sign in', 'verify your',
    # Récompenses
    'you have won', 'congratulations', 'prize', 'reward', 'free gift',
    'click here', 'claim now',
    # FR
    'compte suspendu', 'vérifiez', 'mot de passe', 'carte bancaire',
    'vous avez gagné', 'cliquez ici', 'offre limitée', 'immédiatement',
    # Mobile Money Afrique
    'orange money', 'wave', 'mtn momo', 'free money', 'airtel money',
    'transfert', 'rechargement', 'code secret'
]

def extract_url_features(url: str) -> dict:
    """Extrait 15 features numériques d'une URL."""
    if not isinstance(url, str):
        url = ''
    ext = tldextract.extract(url)
    domain = ext.domain + '.' + ext.suffix if ext.suffix else ext.domain
    return {
        'url_length'         : len(url),
        'domain_length'      : len(domain),
        'num_dots'           : url.count('.'),
        'num_hyphens'        : url.count('-'),
        'num_digits'         : sum(c.isdigit() for c in url),
        'num_special'        : sum(c in '@?=&%#!' for c in url),
        'has_ip'             : int(bool(re.search(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', url))),
        'has_at_sign'        : int('@' in url),
        'has_https'          : int(url.startswith('https')),
        'subdomain_count'    : len(ext.subdomain.split('.')) if ext.subdomain else 0,
        'path_depth'         : url.count('/') - 2,
        'has_suspicious_tld' : int(ext.suffix in ['xyz','tk','ml','cf','ga','gq','pw','top','click','link']),
        'double_slash'       : int('//' in url[8:]),
        'url_entropy'        : (
            -sum((url.count(c)/len(url))*np.log2(url.count(c)/len(url))
                 for c in set(url) if url.count(c)>0)
            if url else 0
        ),
        'has_brand_mismatch' : int(any(
            brand in url.lower() and ext.domain.lower() not in brand
            for brand in ['paypal','amazon','google','facebook','apple','microsoft','orange','wave']
        ))
    }

def extract_text_features(text: str) -> dict:
    """Extrait des features textuelles de contenu."""
    if not isinstance(text, str):
        text = ''
    text_lower = text.lower()
    tokens = word_tokenize(text_lower) if text else []
    words  = [t for t in tokens if t.isalpha() and t not in STOP]
    return {
        'text_length'           : len(text),
        'word_count'            : len(words),
        'phishing_keyword_count': sum(kw in text_lower for kw in PHISHING_KEYWORDS),
        'exclamation_count'     : text.count('!'),
        'question_count'        : text.count('?'),
        'caps_ratio'            : sum(c.isupper() for c in text) / max(len(text), 1),
        'digit_ratio'           : sum(c.isdigit() for c in text) / max(len(text), 1),
        'unique_word_ratio'     : len(set(words)) / max(len(words), 1),
    }

print('✅ Fonctions de feature engineering définies.')

In [ ]:
# ─── 2.1  Construction du dataset texte (FakeNewsNet → NLP) ──────────────────
print('🔄 Extraction des features texte sur FakeNewsNet...')

text_features = fnn['title'].apply(extract_text_features).apply(pd.Series)
url_features  = fnn['url'].apply(extract_url_features).apply(pd.Series)

df_nlp = pd.concat([
    fnn[['title', 'url', 'label_fnn']].rename(columns={'label_fnn': 'label'}),
    text_features,
    url_features
], axis=1).dropna(subset=['label'])

print(f'  ✅ Dataset NLP : {len(df_nlp)} entrées, {df_nlp.shape[1]} colonnes')

In [ ]:
# ─── 2.2  Construction du dataset structurel (UCI → Random Forest) ───────────
print('🔄 Préparation du dataset UCI Phishing...')

# Remplacer les valeurs -1 (qui signifient "suspect" dans UCI) par 0.5
# Les valeurs UCI : -1=phishing, 0=suspicious, 1=legitimate
df_struct = uci.copy()

# Normaliser : on remplace -1 par 0 pour les features (pas le label)
feature_cols = [c for c in df_struct.columns if c != 'label_phishing']
df_struct[feature_cols] = df_struct[feature_cols].replace(-1, 0)
df_struct = df_struct.dropna()

X_struct = df_struct[feature_cols].values
y_struct = df_struct['label_phishing'].values

print(f'  ✅ Dataset structurel : {X_struct.shape[0]} entrées, {X_struct.shape[1]} features')
print(f'  Distribution : {np.bincount(y_struct.astype(int))}')

## 🌲 ÉTAPE 3 — Modèle 1 : Random Forest (Features Structurelles UCI)

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, accuracy_score
)
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns

# ── Split ─────────────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_struct, y_struct, test_size=0.2, random_state=42, stratify=y_struct
)

# ── Rééquilibrage SMOTE si nécessaire ─────────────────────────────────────────
counts = np.bincount(y_train.astype(int))
if max(counts) / min(counts) > 1.5:
    print('⚖️  Déséquilibre détecté → application de SMOTE...')
    sm = SMOTE(random_state=42)
    X_train, y_train = sm.fit_resample(X_train, y_train)
    print(f'   Après SMOTE : {np.bincount(y_train.astype(int))}')

# ── Entraînement Random Forest ────────────────────────────────────────────────
print('\n🌲 Entraînement Random Forest...')
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)

# ── Évaluation ────────────────────────────────────────────────────────────────
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print('\n📊 === Résultats Random Forest ===')
print(f'  Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'  F1-Score : {f1_score(y_test, y_pred):.4f}')
print(f'  AUC-ROC  : {roc_auc_score(y_test, y_proba):.4f}')
print('\n', classification_report(y_test, y_pred, target_names=['Légitime', 'Phishing']))

In [ ]:
# ── Visualisations ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Légitime','Phishing'],
            yticklabels=['Légitime','Phishing'], ax=axes[0])
axes[0].set_title('Matrice de Confusion — Random Forest')
axes[0].set_xlabel('Prédit'); axes[0].set_ylabel('Réel')

# Feature importance (top 15)
importances = pd.Series(rf.feature_importances_, index=feature_cols).nlargest(15)
importances.sort_values().plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top 15 Features — Random Forest')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('data/processed/rf_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graphiques sauvegardés.')

## 🤖 ÉTAPE 4 — Modèle 2 : DistilBERT Fine-tuné (Analyse Textuelle)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split as tts

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU : {torch.cuda.get_device_name(0)}')

MODEL_NAME = 'distilbert-base-multilingual-cased'  # Multilingue FR+EN+AR
MAX_LEN    = 128
BATCH_SIZE = 32
EPOCHS     = 3
LR         = 2e-5

In [ ]:
# ── Préparation du texte ──────────────────────────────────────────────────────
# On combine title + URL pour donner plus de contexte au modèle
df_bert = df_nlp[['title', 'url', 'label']].copy()
df_bert['text'] = (
    df_bert['title'].fillna('') + ' [SEP] ' + df_bert['url'].fillna('')
)
df_bert = df_bert.dropna(subset=['text', 'label'])
df_bert['label'] = df_bert['label'].astype(int)

# Split train/val/test
X_text = df_bert['text'].values
y_text = df_bert['label'].values

X_tr, X_tmp, y_tr, y_tmp = tts(X_text, y_text, test_size=0.3, random_state=42, stratify=y_text)
X_val, X_te, y_val, y_te  = tts(X_tmp,  y_tmp,  test_size=0.5, random_state=42, stratify=y_tmp)

print(f'  Train : {len(X_tr)} | Val : {len(X_val)} | Test : {len(X_te)}')

In [ ]:
# ── Tokenizer & Dataset PyTorch ───────────────────────────────────────────────
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class PhishingDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding='max_length',
            max_length=max_len, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids'      : self.encodings['input_ids'][idx],
            'attention_mask' : self.encodings['attention_mask'][idx],
            'labels'         : self.labels[idx]
        }

train_ds = PhishingDataset(X_tr,  y_tr,  tokenizer, MAX_LEN)
val_ds   = PhishingDataset(X_val, y_val, tokenizer, MAX_LEN)
test_ds  = PhishingDataset(X_te,  y_te,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ Dataloaders créés — {len(train_loader)} batches d\'entraînement')

In [ ]:
# ── Modèle DistilBERT ─────────────────────────────────────────────────────────
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

print(f'✅ Modèle {MODEL_NAME} chargé — {sum(p.numel() for p in model.parameters()):,} paramètres')

In [ ]:
# ── Boucle d'entraînement ─────────────────────────────────────────────────────
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()
history = {'train_loss': [], 'val_loss': [], 'val_f1': []}
best_f1 = 0.0

def evaluate(loader):
    model.eval()
    all_preds, all_labels, total_loss = [], [], 0.0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_preds)
    return avg_loss, f1, auc

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for step, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        train_loss += loss.item()

        if (step + 1) % 20 == 0:
            print(f'  Epoch {epoch+1} | Step {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_train_loss = train_loss / len(train_loader)
    val_loss, val_f1, val_auc = evaluate(val_loader)

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)

    print(f'\n🔵 Epoch {epoch+1}/{EPOCHS}')
    print(f'   Train Loss : {avg_train_loss:.4f}')
    print(f'   Val Loss   : {val_loss:.4f}  |  Val F1 : {val_f1:.4f}  |  Val AUC : {val_auc:.4f}')

    # Sauvegarde du meilleur modèle
    if val_f1 > best_f1:
        best_f1 = val_f1
        model.save_pretrained('models/distilbert_fati_best')
        tokenizer.save_pretrained('models/distilbert_fati_best')
        print(f'   💾 Meilleur modèle sauvegardé (F1={best_f1:.4f})')

print('\n✅ Entraînement terminé !')

In [ ]:
# ── Courbes d'apprentissage ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'],   label='Val Loss',   marker='s')
axes[0].set_title('Courbe de Loss — DistilBERT FATI')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history['val_f1'], label='Val F1', color='green', marker='o')
axes[1].axhline(y=best_f1, linestyle='--', color='red', alpha=0.5, label=f'Best F1={best_f1:.4f}')
axes[1].set_title('F1-Score Validation — DistilBERT FATI')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1-Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('data/processed/bert_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Évaluation finale sur le Test Set ─────────────────────────────────────────
best_model = DistilBertForSequenceClassification.from_pretrained(
    'models/distilbert_fati_best'
).to(DEVICE)

_, test_f1, test_auc = evaluate(test_loader)
print(f'\n🏆 === Résultats Finaux DistilBERT FATI ===')
print(f'   Test F1-Score : {test_f1:.4f}')
print(f'   Test AUC-ROC  : {test_auc:.4f}')

## 📤 ÉTAPE 5 — Export ONNX des Deux Modèles

In [ ]:
# ─── 5.1  Export Random Forest → ONNX ────────────────────────────────────────
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnx

print('📦 Export Random Forest → ONNX...')

initial_type = [('float_input', FloatTensorType([None, X_struct.shape[1]]))]
onnx_rf = convert_sklearn(
    rf,
    initial_types=initial_type,
    options={id(rf): {'zipmap': False}},  # Sortie tableau, pas dict
    target_opset=17
)

rf_onnx_path = 'models/onnx/fati_random_forest.onnx'
with open(rf_onnx_path, 'wb') as f:
    f.write(onnx_rf.SerializeToString())

print(f'  ✅ Random Forest exporté : {rf_onnx_path}')
print(f'  📏 Taille du fichier : {os.path.getsize(rf_onnx_path) / 1024:.1f} KB')

In [ ]:
# ─── 5.2  Export DistilBERT → ONNX (via Optimum) ─────────────────────────────
from optimum.onnxruntime import ORTModelForSequenceClassification
from optimum.onnxruntime.configuration import OptimizationConfig, ORTConfig

print('📦 Export DistilBERT → ONNX via Optimum...')

ort_model = ORTModelForSequenceClassification.from_pretrained(
    'models/distilbert_fati_best',
    export=True
)

bert_onnx_dir = 'models/onnx/fati_distilbert'
ort_model.save_pretrained(bert_onnx_dir)
tokenizer.save_pretrained(bert_onnx_dir)

bert_onnx_path = os.path.join(bert_onnx_dir, 'model.onnx')
print(f'  ✅ DistilBERT exporté : {bert_onnx_path}')
print(f'  📏 Taille du fichier : {os.path.getsize(bert_onnx_path) / (1024*1024):.1f} MB')

In [ ]:
# ─── 5.3  Quantification INT8 (réduction de taille pour le navigateur) ────────
from onnxruntime.quantization import quantize_dynamic, QuantType

print('⚡ Quantification INT8 du modèle DistilBERT...')

bert_quantized_path = 'models/onnx/fati_distilbert_int8.onnx'
quantize_dynamic(
    model_input=bert_onnx_path,
    model_output=bert_quantized_path,
    weight_type=QuantType.QInt8
)

orig_size = os.path.getsize(bert_onnx_path) / (1024*1024)
quant_size = os.path.getsize(bert_quantized_path) / (1024*1024)
print(f'  ✅ Modèle quantifié')
print(f'  📏 Original  : {orig_size:.1f} MB')
print(f'  📏 Quantifié : {quant_size:.1f} MB  (réduction: {(1 - quant_size/orig_size)*100:.1f}%)')

In [ ]:
# ─── 5.4  Validation des modèles ONNX exportés ────────────────────────────────
import onnxruntime as ort

print('🔍 Validation du modèle Random Forest ONNX...')

# Test RF ONNX
sess_rf = ort.InferenceSession(rf_onnx_path)
sample_input = X_test[:5].astype(np.float32)
rf_onnx_preds = sess_rf.run(None, {'float_input': sample_input})[0]
rf_orig_preds = rf.predict(X_test[:5])

print(f'  Prédictions ONNX    : {rf_onnx_preds}')
print(f'  Prédictions sklearn : {rf_orig_preds}')
assert np.array_equal(rf_onnx_preds, rf_orig_preds), '❌ Divergence ONNX/sklearn !'
print('  ✅ Random Forest ONNX validé — résultats identiques')

# Test BERT ONNX
print('\n🔍 Validation du modèle DistilBERT ONNX...')
sess_bert = ort.InferenceSession(bert_quantized_path)
sample_text = ['Your account has been suspended. Verify immediately!', 'Latest news from BBC today']
enc = tokenizer(sample_text, return_tensors='np', padding='max_length',
                truncation=True, max_length=MAX_LEN)
bert_outs = sess_bert.run(None, {
    'input_ids': enc['input_ids'].astype(np.int64),
    'attention_mask': enc['attention_mask'].astype(np.int64)
})
import scipy.special
probs = scipy.special.softmax(bert_outs[0], axis=1)
print(f'  Texte 1 (phishing attendu)  → P(phishing)={probs[0][1]:.3f}')
print(f'  Texte 2 (légitime attendu)  → P(phishing)={probs[1][1]:.3f}')
print('  ✅ DistilBERT ONNX validé')

## 💾 ÉTAPE 6 — Sauvegarde & Téléchargement

In [ ]:
import shutil
import json

# ── Créer un dossier de livraison propre ──────────────────────────────────────
DELIVERY = 'fati_models_delivery'
os.makedirs(DELIVERY, exist_ok=True)

# Copie des modèles
shutil.copy(rf_onnx_path, f'{DELIVERY}/fati_random_forest.onnx')
shutil.copy(bert_quantized_path, f'{DELIVERY}/fati_distilbert_int8.onnx')
shutil.copy('data/processed/rf_evaluation.png',    f'{DELIVERY}/rf_evaluation.png')
shutil.copy('data/processed/bert_training_curves.png', f'{DELIVERY}/bert_training_curves.png')

# Sauvegarde du vocabulaire tokenizer (nécessaire côté JS)
shutil.copy(f'{bert_onnx_dir}/tokenizer.json',     f'{DELIVERY}/tokenizer.json')
shutil.copy(f'{bert_onnx_dir}/tokenizer_config.json', f'{DELIVERY}/tokenizer_config.json')
shutil.copy(f'{bert_onnx_dir}/vocab.txt',          f'{DELIVERY}/vocab.txt')

# Sauvegarde des feature names pour RF
with open(f'{DELIVERY}/rf_feature_names.json', 'w') as f:
    json.dump(feature_cols, f, indent=2)

# Sauvegarde des métriques
metrics = {
    'random_forest': {
        'accuracy': float(accuracy_score(y_test, y_pred)),
        'f1_score': float(f1_score(y_test, y_pred)),
        'auc_roc' : float(roc_auc_score(y_test, y_proba))
    },
    'distilbert': {
        'best_val_f1': float(best_f1),
        'test_f1'    : float(test_f1),
        'test_auc'   : float(test_auc)
    },
    'models': {
        'rf_size_kb'          : round(os.path.getsize(rf_onnx_path) / 1024, 1),
        'bert_quantized_mb'   : round(os.path.getsize(bert_quantized_path) / (1024*1024), 1)
    }
}
with open(f'{DELIVERY}/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('📋 Métriques finales :')
print(json.dumps(metrics, indent=2))

# Zip et téléchargement
shutil.make_archive('fati_models_delivery', 'zip', DELIVERY)
print('\n📦 Archive créée : fati_models_delivery.zip')

In [ ]:
# ── Téléchargement depuis Colab ────────────────────────────────────────────────
try:
    from google.colab import files
    files.download('fati_models_delivery.zip')
    print('⬇️  Téléchargement lancé !')
except ImportError:
    print('ℹ️  Pas dans Colab — fichier disponible dans le répertoire courant.')
    print(f'   Chemin : {os.path.abspath("fati_models_delivery.zip")}')

---
## ✅ Récapitulatif — Ce que tu obtiens

| Fichier | Description | Usage dans l'extension |
|---|---|---|
| `fati_random_forest.onnx` | Modèle RF sur features structurelles UCI | Analyse URL + DOM (rapide, <50ms) |
| `fati_distilbert_int8.onnx` | DistilBERT multilingue quantifié | Analyse textuelle NLP (<800ms) |
| `tokenizer.json` | Vocabulaire pour tokenisation JS | Requis pour BERT côté navigateur |
| `rf_feature_names.json` | Noms des features RF dans l'ordre | Pour construire le vecteur d'entrée |
| `metrics.json` | Performance des modèles | Documentation / README |

### 🔌 Intégration dans l'extension Chrome :
```
fati-extension/src/models/
├── fati_random_forest.onnx       # → chargé via onnxruntime-web
├── fati_distilbert_int8.onnx     # → chargé via onnxruntime-web
├── tokenizer.json                # → tokenisation locale
└── rf_feature_names.json         # → mapping features
```

### 📚 Datasets utilisés :
- **FakeNewsNet** (KaiDMML, ASU) : Fake/Real news — PolitiFact + GossipCop
- **UCI Phishing Websites Dataset** : 11 055 sites, 30 features structurelles

---
*FATI — iSAFE Hackathon SMSI 2026 — Mouhammadou DIA*